# 🏥 MaternaCare: Model 5 — Optical Character Recognition (OCR) Microservice
### Complete Full-Text Multi-Page Medical Document Extraction (EasyOCR + PyPDF + PyTesseract)

This Colab notebook hosts **Model 5 (OCR Model)** from the MaternaCare Distributed Architecture.
It accepts any medical PDF (even 5–10+ scanned pages or photos) and extracts **100% of every line, word, and table**, outputting a complete Markdown (`.md`) document back to your Next.js app.

### 1. Install Dependencies & System OCR Packages

In [ ]:
!pip install -q fastapi uvicorn pyngrok python-multipart pypdf pdf2image pytesseract pillow easyocr
!apt-get install -y -qq poppler-utils tesseract-ocr tesseract-ocr-eng

### 2. Run the OCR Microservice with Ngrok Public Tunnel

In [ ]:
import io
import os
import threading
import uvicorn
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pypdf import PdfReader
from PIL import Image
import pdf2image
import pytesseract
import easyocr
from pyngrok import ngrok

app = FastAPI(title="MaternaCare Model 5 OCR")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

print("⏳ Initializing EasyOCR engine on GPU...")
reader = easyocr.Reader(['en'], gpu=True)
print("✅ EasyOCR Ready!")

@app.get("/")
def root():
    return {"status": "online", "model": "MaternaCare Model 5 (Full-Text OCR)"}

@app.post("/api/ocr")
async def extract_ocr(file: UploadFile = File(...)):
    contents = await file.read()
    filename = file.filename or "document.pdf"
    extracted_pages = []
    is_pdf = filename.lower().endswith(".pdf") or (file.content_type and "pdf" in file.content_type)

    if is_pdf:
        # 1. Try digital text extraction first
        try:
            pdf_obj = io.BytesIO(contents)
            reader_pdf = PdfReader(pdf_obj)
            total_pages = len(reader_pdf.pages)
            for idx, page in enumerate(reader_pdf.pages):
                txt = page.extract_text()
                if txt and len(txt.strip()) > 30:
                    extracted_pages.append(f"### 📄 Page {idx + 1} of {total_pages}\n\n{txt.strip()}")
        except Exception as e:
            print(f"Digital PDF error: {e}")

        # 2. If scanned images in PDF, convert to image and run Optical Character Recognition
        if len(extracted_pages) == 0:
            try:
                images = pdf2image.convert_from_bytes(contents)
                for idx, img in enumerate(images):
                    img_byte_arr = io.BytesIO()
                    img.save(img_byte_arr, format='PNG')
                    ocr_lines = reader.readtext(img_byte_arr.getvalue(), detail=0)
                    page_text = "\n".join(ocr_lines).strip()
                    if len(page_text) < 40:
                        page_text = pytesseract.image_to_string(img).strip()
                    if page_text:
                        extracted_pages.append(f"### 📄 Page {idx + 1} of {len(images)} (Optical OCR)\n\n{page_text}")
            except Exception as e2:
                print(f"Raster scan error: {e2}")
    else:
        # Image file
        img = Image.open(io.BytesIO(contents))
        ocr_lines = reader.readtext(contents, detail=0)
        page_text = "\n".join(ocr_lines).strip()
        if len(page_text) < 40:
            page_text = pytesseract.image_to_string(img).strip()
        extracted_pages.append(f"### 📄 Scanned Image Document\n\n{page_text}")

    full_text = "\n\n".join(extracted_pages)
    markdown_doc = f"# 📄 COMPLETE EXTRACTED MEDICAL DOCUMENT\n**File:** `{filename}`  \n**Pages Extracted:** {len(extracted_pages)}  \n\n---\n\n{full_text}\n\n---\n*Extracted via MaternaCare Model 5 OCR Engine.*"

    return {
        "success": True,
        "filename": filename,
        "pages_count": len(extracted_pages),
        "markdown": markdown_doc,
        "text": full_text
    }

# Enter your ngrok authtoken (Get free from https://dashboard.ngrok.com/get-started/your-authtoken)
NGROK_AUTHTOKEN = "" # Paste your token here
if NGROK_AUTHTOKEN:
    ngrok.set_auth_token(NGROK_AUTHTOKEN)

# Start public tunnel
tunnel = ngrok.connect(8000)
print("\n" + "="*70)
print(f"🔗 PUBLIC COLAB OCR URL: {tunnel.public_url}")
print(f"👉 Add to your Next.js .env.local: COLAB_OCR_URL={tunnel.public_url}")
print("="*70 + "\n")

# Run Uvicorn server
uvicorn.run(app, host="0.0.0.0", port=8000)